In [ ]:
!pip install PyPortfolioOpt
!pip install yfinance
!pip install --upgrade yfinance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pypfopt import EfficientFrontier, risk_models, expected_returns
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go
from datetime import date
from dateutil.relativedelta import relativedelta
import cvxpy as cp
import warnings
warnings.filterwarnings('ignore')
import matplotlib.cm as cm
from datetime import datetime, timedelta
from pypfopt.exceptions import OptimizationError

In [ ]:
risco = ["^IRX"]  # TREASURY BILL de 13 semanas (3 meses)
end_date_fr = datetime.now().date()
start_date_fr = end_date_fr - timedelta(days=120) #datetime(2025,6,20)

print(f"Baixando dados de {start_date_fr} até {end_date_fr}...")
prices_raw_fr = yf.download(risco, start=start_date_fr, end=end_date_fr, interval="1h", progress=False, auto_adjust=False)

if 'Close' in prices_raw_fr:
    if not prices_raw_fr.empty:
        risk_free_rate = float(prices_raw_fr['Close'].mean() / 100)
    else:
        print("Nenhum dado encontrado para IRX no período especificado. Usando risk_free_rate default.")
        risk_free_rate = 0.0412
else:
    print("IRX sem coluna Close. Usando risk_free_rate default.")

In [ ]:
risk_free_rate

In [ ]:
# =============================================================================
# CONFIGURAÇÕES E DADOS
# =============================================================================

class CryptoPortfolioAnalyzer:
    def __init__(self, tickers, start_date=None, end_date=None):
        self.tickers = tickers
        self.end_date =  datetime.now().date()
        self.start_date = self.end_date - timedelta(days=120)#datetime(2025,6,20)self.end_date - timedelta(days=120)
        self.prices = None
        self.returns = None
        self.benchmark_ticker = "BTC-USD"
        
    def download_data(self):
        """Download e preparação dos dados"""
        print(f"Baixando dados de {self.start_date} até {self.end_date}...")
        try:
            prices_raw = yf.download(self.tickers, start=self.start_date, end=self.end_date, interval="1h", progress=False, auto_adjust=False)
            self.prices = prices_raw['Close'] if len(self.tickers) > 1 else pd.DataFrame({self.tickers[0]: prices_raw['Close']})
            #self.prices[]


            # Calcula retornos logarítmicos
            self.returns = np.log(self.prices / self.prices.shift(1))

            mediana_retornos = self.returns.median()
            self.returns = self.returns.fillna(mediana_retornos)
            #diplay(self.returns)
            return True
        
        except Exception as e:
            print(f"Erro ao baixar dados: {e}")
            return False
            
    def download_prices_fr(self):
        return self.returns

    def plot_price_evolution(self):
        """Gráfico da evolução normalizada dos preços"""
        normalized_prices = self.prices / self.prices.iloc[0]
        
        plt.figure(figsize=(14, 8))
        for ticker in self.tickers:
            plt.plot(normalized_prices.index, normalized_prices[ticker], 
                    label=ticker, linewidth=2)
        
        plt.title('Evolução Normalizada dos Preços das Criptomoedas',
                 fontsize=16, fontweight='bold')
        plt.xlabel('Data', fontweight='bold')
        plt.ylabel('Preço Normalizado (Base = 1)', fontweight='bold')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
    
    # def plot_returns_distribution(self):
    #     """Análise da distribuição dos retornos"""
    #     fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
        
    #     # Densidade dos retornos
    #     for ticker in self.returns.columns:
    #         sns.kdeplot(data=self.returns[ticker], label=ticker, 
    #                    fill=True, alpha=0.3, ax=ax1)
        
    #     ax1.set_title('Distribuição dos Retornos Diários', fontsize=14, fontweight='bold')
    #     ax1.set_xlabel('Retorno', fontweight='bold')
    #     ax1.set_ylabel('Densidade', fontweight='bold')
    #     ax1.legend()
    #     ax1.grid(True, alpha=0.3)
        

        
    #     # Box plot dos retornos
    #     returns_melted = self.returns.reset_index().melt(
    #         id_vars='Datetime', var_name='Ticker', value_name='Return')
    #     sns.boxplot(data=returns_melted, x='Ticker', y='Return', ax=ax2)
    #     ax2.set_title('Box Plot dos Retornos', fontsize=14, fontweight='bold')
    #     ax2.set_xlabel('Ativo', fontweight='bold')
    #     ax2.set_ylabel('Retorno', fontweight='bold')
    #     ax2.tick_params(axis='x', rotation=45)
        
        
    #     plt.tight_layout()
    #     plt.show()
    def plot_returns_distribution(self):


        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Densidade dos retornos
        for ticker in self.returns.columns:
            sns.kdeplot(
                data=self.returns[ticker],
                label=ticker,
                fill=True,
                alpha=0.4,
                ax=ax1,
                bw_adjust=2.0,     # suaviza e reduz o pico
                common_norm=False  # evita normalização conjunta que exagera a densidade
            )

        ax1.set_title('Distribuição dos Retornos Diários', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Retorno', fontweight='bold')
        ax1.set_ylabel('Densidade', fontweight='bold')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

    # 🔹 Limita o eixo X para ignorar outliers extremos (mais visual)
        ax1.set_xlim(-0.05, 0.05)

    # 🔹 Limita o eixo Y para achatar
        ax1.set_ylim(0, ax1.get_ylim()[1] * 0.4)

    # 🔹 Adiciona uma linha no zero para referência
        ax1.axvline(0, color='black', linestyle='--', alpha=0.6)

    # Box plot dos retornos
        returns_melted = self.returns.reset_index().melt(
            id_vars='Datetime', var_name='Ticker', value_name='Return'
        )

        sns.boxplot(
            data=returns_melted,
            x='Ticker',
            y='Return',
            ax=ax2,
            showfliers=True,
            width=0.6
    )

        ax2.set_title('Box Plot dos Retornos', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Ativo', fontweight='bold')
        ax2.set_ylabel('Retorno', fontweight='bold')
        ax2.tick_params(axis='x', rotation=45)
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()
    


    ###


    def optimize_portfolio(self, objective='sharpe', risk_free_rate=risk_free_rate): ### Mudar o objetivo para ser variavel       prices_fr    
        """Otimização robusta do portfolio com múltiplas estratégias"""
        # Remove benchmark dos ativos para otimização
        assets_returns = self.returns.drop(columns=[self.benchmark_ticker], errors='ignore')
               
        if len(assets_returns.columns) == 0:
            print("Erro: Nenhum ativo disponível para otimização")
            return None, None, None
        
        print(f"Otimizando portfolio com {len(assets_returns.columns)} ativos...")
        print(f"Período de dados: {len(assets_returns)} dias")
        
        ######################### Estratégia 1: Usar Ledoit-Wolf Shrinkage (mais robusto) ###################################
        try:
             print("Tentativa 1: Usando Ledoit-Wolf shrinkage...")
             mu = expected_returns.mean_historical_return(self.prices.drop(columns=[self.benchmark_ticker], errors='ignore'), frequency=8760)
             S = risk_models.CovarianceShrinkage(assets_returns).ledoit_wolf()

            
             # Adiciona regularização manual se necessário
             if np.linalg.cond(S) > 1e12:  # Matriz mal condicionada
                 print("Matriz mal condicionada, aplicando regularização...")
                 regularization = 1e-5 * np.trace(S) / len(S)
                 S += regularization * np.eye(len(S))
            
             ef = EfficientFrontier(mu, S, weight_bounds=(0.01, 0.5))
            
             if objective == 'sharpe':
                 weights = ef.max_sharpe(risk_free_rate=risk_free_rate)
             else:
                 weights = ef.min_volatility()
            
             cleaned_weights = ef.clean_weights(cutoff=0.005)
             performance = ef.portfolio_performance(risk_free_rate=risk_free_rate, verbose=False)
            
             print(" Otimização bem-sucedida com Ledoit-Wolf")
             return cleaned_weights, performance, assets_returns
            
        except Exception as e:
             print(f" Falha na tentativa 1: {e}")
        
        
              
        # Estratégia 2: Covariância exponencial
        try:
            print("Tentativa 2: Usando covariância exponencial...")
            mu = expected_returns.ema_historical_return(assets_returns, frequency=8760)
            S = risk_models.exp_cov(assets_returns, frequency=8760)
            
            # Regularização
            regularization = 1e-4 * np.trace(S) / len(S)
            S += regularization * np.eye(len(S))
            
            ef = EfficientFrontier(mu, S, weight_bounds=(0.02, 0.6))
            weights = ef.min_volatility()  # Mais conservador
            
            cleaned_weights = ef.clean_weights(cutoff=0.01)
            performance = ef.portfolio_performance(risk_free_rate=risk_free_rate, verbose=False)
            
            print(" Otimização bem-sucedida com covariância exponencial")
            return cleaned_weights, performance, assets_returns
            
        except Exception as e:
            print(f" Falha na tentativa 2: {e}")
        
        # Estratégia 3: Método simples baseado em correlação
        try:
            print("Tentativa 3: Usando método baseado em Sharpe simples...")
            
            # Calcula Sharpe ratio individual
            daily_returns = assets_returns.mean()
            daily_vol = assets_returns.std()
            sharpe_ratios = daily_returns / daily_vol
            
            # Normaliza para criar pesos
            sharpe_ratios = sharpe_ratios.fillna(0)
            sharpe_ratios = np.maximum(sharpe_ratios, 0)  # Remove valores negativos
            
            if sharpe_ratios.sum() > 0:
                weights_array = sharpe_ratios / sharpe_ratios.sum()
            else:
                weights_array = np.ones(len(assets_returns.columns)) / len(assets_returns.columns)
            
            # Converte para dicionário
            cleaned_weights = {asset: float(weight) for asset, weight in 
                             zip(assets_returns.columns, weights_array)}
            
            # Calcula performance aproximada
            port_return = np.sum(weights_array * daily_returns) * 8760
            port_vol = np.sqrt(np.dot(weights_array, np.dot(assets_returns.cov() * 8760, weights_array)))
            port_sharpe = port_return / port_vol if port_vol > 0 else 0
            
            performance = (port_return, port_vol, port_sharpe)
            
            print(" Otimização bem-sucedida com método Sharpe simples")
            return cleaned_weights, performance, self.returns
            
        except Exception as e:
            print(f" Falha na tentativa 3: {e}")

    
    def plot_portfolio_allocation(self, weights):
        """Gráfico da alocação do portfolio"""
        weights_df = pd.DataFrame({
            'Asset': list(weights.keys()),
            'Weight': list(weights.values())
        }).sort_values('Weight', ascending=True)
        
        plt.figure(figsize=(10, 6))
        colors = cm.viridis(np.linspace(0, 1, len(weights_df)))
        plt.barh(weights_df['Asset'], weights_df['Weight'], color=colors)
        plt.title('Alocação Otimizada do Portfolio', fontsize=16, fontweight='bold')
        plt.xlabel('Peso (%)', fontweight='bold')
        plt.ylabel('Ativos', fontweight='bold')
        
        # Adiciona valores nas barras
        for i, v in enumerate(weights_df['Weight']):
            plt.text(v + 0.01, i, f'{v:.1%}', va='center', fontweight='bold')
        
        plt.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    def calculate_portfolio_performance(self, weights, assets_returns):
        """Calcula performance do portfolio ao longo do tempo"""
        # Calcula retornos do portfolio
        portfolio_returns = pd.Series(0.0, index=assets_returns.index, dtype=float)
        
        for asset, weight in weights.items():
            if asset in assets_returns.columns:
                portfolio_returns += assets_returns[asset] * weight
        
        # Calcula retornos cumulativos
        #portfolio_cumulative = (1 + portfolio_returns).cumprod()
        portfolio_cumulative = np.exp(portfolio_returns.cumsum())
        
        # Benchmark (se disponível)
        benchmark_cumulative = None
        if self.benchmark_ticker in self.returns.columns:
            #benchmark_returns = self.returns[self.benchmark_ticker]
            #benchmark_cumulative = (1 + benchmark_returns).cumprod()
            benchmark_logret = self.returns[self.benchmark_ticker]
            benchmark_cumulative = np.exp(benchmark_logret.cumsum())
        
        return portfolio_returns, portfolio_cumulative, benchmark_cumulative
    
    def plot_performance_comparison(self, portfolio_cumulative, benchmark_cumulative=None):
        """Gráfico de comparação de performance"""
        plt.figure(figsize=(14, 8))
        
        plt.plot(portfolio_cumulative.index, portfolio_cumulative, 
                label='Portfolio Otimizado', linewidth=2, color='darkblue')
        
        if benchmark_cumulative is not None:
            plt.plot(benchmark_cumulative.index, benchmark_cumulative, 
                    label=f'Benchmark ({self.benchmark_ticker})', 
                    linewidth=2, color='red', alpha=0.7)
        
        plt.title('Performance: Portfolio vs Benchmark', fontsize=16, fontweight='bold')
        plt.xlabel('Data', fontweight='bold')
        plt.ylabel('Retorno Cumulativo', fontweight='bold')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    #-------------começa aqui-----------------
    def generate_efficient_frontier(self, assets_returns, num_points=30):
        """Gera a fronteira eficiente com método robusto"""
        try:
            # 1. Estimar retornos esperados e matriz de covariância
            mu = expected_returns.ema_historical_return(assets_returns, frequency=8760)
            S = risk_models.CovarianceShrinkage(assets_returns).ledoit_wolf()
            
            # 2. Regularização adicional (opcional, mas útil)
            regularization = 1e-4 * np.trace(S) / len(S)
            S += regularization * np.eye(len(S))
            
            # 3. Determinar limites reais de retorno viável
            ef_min = EfficientFrontier(mu, S, weight_bounds=(0.01, 0.8), solver="SCS")
            ef_min.min_volatility()
            min_ret, _, _ = ef_min.portfolio_performance(verbose=False)
            
            ef_max = EfficientFrontier(mu, S, weight_bounds=(0.01, 0.8), solver="SCS")
            ef_max.max_sharpe()
            max_ret, _, _ = ef_max.portfolio_performance(verbose=False)
            
            # Garantir que max_ret > min_ret
            if max_ret <= min_ret:
                print(" Retorno máximo não é maior que o mínimo. Fronteira degenerada.")
                return np.array([]), np.array([])
            
            # 4. Definir alvo de retornos dentro do intervalo viável
            buffer = 0.01 * (max_ret - min_ret)  # pequeno buffer para evitar bordas
            target_returns = np.linspace(min_ret + buffer, max_ret - buffer, num_points)
            
            efficient_portfolios = []
            
            for target in target_returns:
                try:
                    ef = EfficientFrontier(mu, S, weight_bounds=(0.01, 0.8))
                    ef.efficient_return(target_return=target)
                    ret, vol, _ = ef.portfolio_performance(verbose=False)
                    if vol > 0 and np.isfinite(ret) and np.isfinite(vol):
                        efficient_portfolios.append((ret, vol))
                except (OptimizationError, ValueError, RuntimeError) as e:
                    # Log opcional para depuração:
                    # print(f" Falha no retorno alvo {target:.4f}: {e}")
                    continue
            
            if efficient_portfolios:
                returns_ef, volatility_ef = zip(*efficient_portfolios)
                return np.array(returns_ef), np.array(volatility_ef)
            else:
                print(" Nenhum portfólio eficiente foi gerado.")
                return np.array([]), np.array([])
                
        except Exception as e:
            print(f" Erro na geração da fronteira eficiente: {e}")
            return np.array([]), np.array([])
    
    
    def monte_carlo_simulation(self, assets_returns, num_simulations=5000):
        """Simulação Monte Carlo para portfolios aleatórios"""
        np.random.seed(42)
        
        num_assets = len(assets_returns.columns)
        results = np.zeros((4, num_simulations))  # [retorno, volatilidade, sharpe, pesos]
        
        mu = assets_returns.mean() * 8760  # Anualizado
        cov = assets_returns.cov() * 8760  # Anualizado
        
        for i in range(num_simulations):
            # Gera pesos aleatórios
            weights = np.random.random(num_assets)
            weights /= weights.sum()
            
            # Calcula métricas do portfolio
            portfolio_return = np.sum(weights * mu)
            portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov, weights)))
            sharpe_ratio = portfolio_return / portfolio_volatility if portfolio_volatility > 0 else 0
            
            results[0, i] = portfolio_return
            results[1, i] = portfolio_volatility
            results[2, i] = sharpe_ratio
        
        return results
    
    def plot_efficient_frontier_with_simulation(self, assets_returns, optimal_weights):
        """Plota fronteira eficiente com simulação Monte Carlo - versão robusta"""
        # Simulação Monte Carlo
        mc_results = self.monte_carlo_simulation(assets_returns)
        
        # Performance do portfolio otimizado
        try:
            mu = assets_returns.mean() * 8760  # Anualizado
            cov_matrix = assets_returns.cov() * 8760  # Anualizado
            
            weights_array = np.array([optimal_weights.get(asset, 0) for asset in assets_returns.columns])
            
            opt_return = np.sum(weights_array * mu)
            opt_volatility = np.sqrt(np.dot(weights_array, np.dot(cov_matrix, weights_array)))
            
        except Exception as e:
            print(f" Erro no cálculo do portfolio otimizado: {e}")
            opt_return, opt_volatility = 0, 0
        
        plt.figure(figsize=(12, 8))
        
        # Simulação Monte Carlo
        if mc_results.size > 0:
            scatter = plt.scatter(mc_results[1], mc_results[0], 
                                c=mc_results[2], cmap='viridis', 
                                alpha=0.6, s=20)
            plt.colorbar(scatter, label='Sharpe Ratio')
            #plt.plot(fronteira['Riscos'], fronteira['Retornos'], 
            #    color='DarkGreen', linewidth=3, label='Fronteira Eficiente')
        
        # Fronteira eficiente (tentativa)
        try:
            ef_returns, ef_volatility = self.generate_efficient_frontier(assets_returns)
            if len(ef_returns) > 0:
                plt.plot(ef_volatility, ef_returns, 'r-', linewidth=3, 
                        label='Fronteira Eficiente')
        except:
            print(" Fronteira eficiente não disponível")

       
        
        # Portfolio otimizado
        if opt_volatility > 0 and opt_return > 0:
            plt.scatter(opt_volatility, opt_return, color='red', 
                       s=200, marker='*', label='Portfolio Otimizado', 
                       edgecolors='black', linewidth=2)
        
        plt.title('Análise de Portfolio: Simulação Monte Carlo', 
                 fontsize=16, fontweight='bold')
        plt.xlabel('Volatilidade (Risco)', fontweight='bold')
        plt.ylabel('Retorno Esperado', fontweight='bold')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    def print_performance_summary(self, weights, performance, portfolio_returns):
        """Imprime resumo da performance"""
        print("=" * 60)
        print("RESUMO DA ANÁLISE DO PORTFOLIO")
        print("=" * 60)
        
        # Informações do portfolio
        print(f"\n COMPOSIÇÃO DO PORTFOLIO:")
        for asset, weight in sorted(weights.items(), key=lambda x: x[1], reverse=True):
            if weight > 0.01:  # Apenas pesos > 1%
                print(f"   {asset}: {weight:.1%}")
        
        print(f"\n PERFORMANCE ESPERADA (Anualizada):")
        print(f"   Retorno Esperado: {performance[0]:.2%}")
        print(f"   Volatilidade: {performance[1]:.2%}")
        print(f"   Sharpe Ratio: {performance[2]:.3f}")
        
        # Estatísticas dos retornos realizados
        print(f"\n ESTATÍSTICAS DOS RETORNOS (Período):")
        print(f"   Retorno Médio Diário: {portfolio_returns.mean():.4%}")
        print(f"   Volatilidade Diária: {portfolio_returns.std():.4%}")
        print(f"   Melhor Dia: {portfolio_returns.max():.2%}")
        print(f"   Pior Dia: {portfolio_returns.min():.2%}")
        
        # Métricas de risco
        var_95 = np.percentile(portfolio_returns, 5)
        print(f"   VaR 95%: {var_95:.2%}")

In [ ]:
def main():
    # Configuração
    tickers = ["BTC-USD","ETH-USD", "BNB-USD", "XRP-USD", "SOL-USD", "DOGE-USD", "ADA-USD", "LINK-USD", "XLM-USD", "BCH-USD", "LEO-USD"] 

    # Inicializa analisador
    analyzer = CryptoPortfolioAnalyzer(tickers)

    # Download dos dados
    if not analyzer.download_data():
        return

    # Análises exploratórias
    print("\n1. Gerando gráficos exploratórios...")
    analyzer.plot_price_evolution()
    analyzer.plot_returns_distribution()
    
    # Otimização do portfolio
    print("\n2. Otimizando portfolio...")
    weights, performance, assets_returns = analyzer.optimize_portfolio(objective='sharpe')
    
    if weights is None:
        print("Erro na otimização. Encerrando análise.")
        return
    
    # Visualizações da otimização
    print("\n3. Gerando visualizações do portfolio otimizado...")
    analyzer.plot_portfolio_allocation(weights)
    
    # Análise de performance
    print("\n4. Calculando performance...")
    portfolio_returns, portfolio_cumulative, benchmark_cumulative = analyzer.calculate_portfolio_performance(
        weights, assets_returns)
    
    analyzer.plot_performance_comparison(portfolio_cumulative, benchmark_cumulative)
    
    # Fronteira eficiente
    print("\n5. Gerando fronteira eficiente...")
    analyzer.plot_efficient_frontier_with_simulation(assets_returns, weights)
    
    # Resumo final
    analyzer.print_performance_summary(weights, performance, portfolio_returns)
    
    print("\n Análise concluída com sucesso!")

if __name__ == "__main__":
    main()

In [ ]:
# Investimos $99.900 porque houve arredondamento no percentual. Para não perde a conta toda, inestimos o mesmo valor em bitcoin.

In [ ]:
import random
tickers_aleatoria =  ["WBTC-USD", "WBETH-USD", "AVAX-USD", "BCH-USD", "LTC-USD", "BTCB-USD", "CRO-USD", "BCH-USD", "LTC-USD", "SHIB-USD","LTC-USD", "CRO-USD", "AVAX-USD", "LTC-USD", "BCH-USD", "DOT-USD", "NEAR-USD", "SHIB-USD", "TRX-USD", "ETC-USD", "ICP-USD", "APT-USD", "FIL-USD", "HBAR-USD", "ATOM-USD", "VET-USD", "OP-USD", "INJ-USD", "AAVE-USD", "QNT-USD", "MKR-USD", "RUNE-USD", "FLOW-USD", "XTZ-USD", "KAS-USD", "EGLD-USD", "CHZ-USD", "MANA-USD", "ENJ-USD", "SAND-USD", "AXS-USD", "GALA-USD", "CRO-USD", "LDO-USD", "RPL-USD", "YFI-USD", "ZRX-USD", "UMA-USD", "REN-USD", "FET-USD", "OCEAN-USD", "HNT-USD", "FLOKI-USD", "KNC-USD", "LEND-USD", "TWT-USD", "SUSHI-USD", "BAL-USD", "CRV-USD", "SPELL-USD", "YGG-USD", "BADGER-USD", "KEEP-USD", "CELO-USD", "BNT-USD", "MITH-USD"]


amostra = random.sample(tickers_aleatoria, 10)

In [ ]:
amostra